# DriveBuddyAI Machine Learning Challenge: Text-Based Pav Bhaji Classifier
**Exploratory Data Analysis, Feature Engineering, Model Benchmarking & Evaluation**

This notebook demonstrates the end-to-end data science and machine learning workflow for predicting whether an Instagram post is about **Pav Bhaji** (Class 1) or **Not Pav Bhaji** (Class 0) strictly using post metadata and textual features without computer vision.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 150

# Add project source to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader import load_raw_dataset, create_train_test_splits
from src.preprocessing import clean_text, TextCleanerTransformer
from src.feature_engineering import get_combined_feature_union, MetadataFeatureExtractor
from src.train import train_and_compare_models
from src.predict import predict_post, load_trained_model

## 1. Data Loading & Schema Exploration
We load `pavbhaji.json` and cross-reference filenames with `images/0` (non-pav bhaji) and `images/1` (pav bhaji).

In [ ]:
df_labeled, df_all = load_raw_dataset(base_dir=project_root)
print(f"Total JSON records: {len(df_all)}")
print(f"Labeled records: {len(df_labeled)}")
print(f"Class Distribution:\n{df_labeled['label'].value_counts()}")
df_labeled[['id', 'image_id', 'label', 'likes', 'comments_count', 'caption']].head(5)

## 2. Text Preprocessing & Compound Hashtag Splitting
Non-Pav Bhaji posts often attach `#pavbhaji` alongside other dishes (`#chicken`, `#pasta`, `#dosa`, `#pakoda`). We decompose compound tags and clean noise while preserving food vocabulary.

In [ ]:
sample_raw = "TAG A PAV BHAJI FANATIC #cheesepavbhaji #mumbaifoodie at @sardarpavbhaji http://bit.ly/pb"
sample_clean = clean_text(sample_raw)
print("Raw Text:    ", sample_raw)
print("Cleaned Text:", sample_clean)

## 3. Train/Test Stratified Split & 5-Fold Model Benchmarking

In [ ]:
train_df, test_df = create_train_test_splits(df_labeled, test_size=0.2, random_state=42)
results, best_model_name, best_pipeline = train_and_compare_models(train_df, test_df)

summary_rows = []
for name, res in results.items():
    summary_rows.append({
        'Model': name,
        'CV F1': f"{res['cv_metrics']['cv_f1_mean']:.4f} ± {res['cv_metrics']['cv_f1_std']:.4f}",
        'Test Accuracy': f"{res['test_accuracy']:.4f}",
        'Test Precision': f"{res['test_precision']:.4f}",
        'Test Recall': f"{res['test_recall']:.4f}",
        'Test F1-Score': f"{res['test_f1']:.4f}",
        'Test ROC-AUC': f"{res['test_roc_auc']:.4f}"
    })
pd.DataFrame(summary_rows)

## 4. Inference & Interactive Prediction

In [ ]:
sample_test_post = {
    'id': 'demo_sample_101',
    'edge_media_to_caption': {'edges': [{'node': {'text': 'Delicious steaming hot Pav Bhaji with generous dollops of Amul butter at Juhu Beach!'}}]}
}
pred_res = predict_post(sample_test_post, model=best_pipeline)
print(json.dumps(pred_res, indent=2))